<a href="https://colab.research.google.com/github/rooparajprojects/RoopaBhavani/blob/main/MultiAgentGoogleADK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries ---
import os
import sys
import json
import asyncio
import random
import string
from uuid import uuid4
from typing import Any, List

import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, Markdown, display

# --- ADK, Agent, and Evaluation Components ---
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content, Part


print("✅ All libraries are ready to go!")



✅ All libraries are ready to go!


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [3]:
# --- API Key Configuration ---
from google.colab import userdata

# Option 1: Use Colab Secrets (recommended)
# Go to the 🔑 icon in the left sidebar, add a secret named GOOGLE_API_KEY
try:
    GOOGLE_API_KEY = userdata.get('GoogleADKMultiAgent')
    print("✅ API key loaded from Colab Secrets.")
except Exception:
    # Option 2: Paste it directly (less secure but fine for learning)
    import getpass
    GOOGLE_API_KEY = getpass.getpass("🔑 Enter your Google AI Studio API key: ")
    print("✅ API key entered manually.")


✅ API key loaded from Colab Secrets.


In [4]:
# --- Set Environment Variables for ADK ---

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

print(f"✅ API key configured (starts with '{GOOGLE_API_KEY[:6]}...')")
print("✅ Using Google AI Studio (not Vertex AI).")


✅ API key configured (starts with 'AQ.Ab8...')
✅ Using Google AI Studio (not Vertex AI).


In [5]:
# --- Agent Definition ---

def create_day_trip_agent():
    """Create the Spontaneous Day Trip Generator agent"""
    return Agent(
        name="day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
        instruction="""
        You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

        Your Mission:
        Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
        """,
        tools=[google_search]
    )

day_trip_agent = create_day_trip_agent()
print(f"🧞 Agent '{day_trip_agent.name}' is created and ready for adventure!")

🧞 Agent 'day_trip_agent' is created and ready for adventure!


In [6]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [7]:
# --- Let's test the Day Trip Genie! ---

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

🗣️ User Query: 'Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!'

🚀 Running query for agent: 'day_trip_agent' in session: '7eb14b52-e79e-4fbf-a576-5d402c42fb00'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Here's a relaxing and artsy day trip itinerary near Sunnyvale, CA, designed to be affordable for a spontaneous Tuesday outing!

---

## Relaxing & Artsy Day Trip: Sunnyvale & Palo Alto (Tuesday, June 16, 2026)

This itinerary focuses on free or low-cost activities, combining artistic appreciation with natural beauty and relaxation.

### ☀️ Morning (10:00 AM - 1:00 PM): Architectural & Outdoor Art Appreciation

**Activity: Explore Stanford University Campus**
*   **Location:** Stanford University, Palo Alto, CA
*   **Cost:** Free (excluding parking)
*   **Details:** While many indoor museums are closed on Tuesdays, Stanford's beautiful campus offers a wealth of architectural wonders and outdoor art that are pe

Here's a relaxing and artsy day trip itinerary near Sunnyvale, CA, designed to be affordable for a spontaneous Tuesday outing!

---

## Relaxing & Artsy Day Trip: Sunnyvale & Palo Alto (Tuesday, June 16, 2026)

This itinerary focuses on free or low-cost activities, combining artistic appreciation with natural beauty and relaxation.

### ☀️ Morning (10:00 AM - 1:00 PM): Architectural & Outdoor Art Appreciation

**Activity: Explore Stanford University Campus**
*   **Location:** Stanford University, Palo Alto, CA
*   **Cost:** Free (excluding parking)
*   **Details:** While many indoor museums are closed on Tuesdays, Stanford's beautiful campus offers a wealth of architectural wonders and outdoor art that are perfect for a relaxing, artsy stroll. Enjoy the stunning Romanesque Revival architecture, particularly around the **Main Quad** and **Memorial Church**. Wander through the **Rodin Sculpture Garden**, which features a significant collection of Auguste Rodin's bronzes, accessible outdoors even when the Cantor Arts Center is closed. You can spend your time sketching, taking photographs, or simply enjoying the serene atmosphere and impressive surroundings.
*   **Parking:** Parking is enforced Monday through Friday, 8:00 AM to 4:00 PM, and typically requires payment via the ParkMobile app. Be sure to check Stanford's parking website for current rates and instructions.

### 🧺 Afternoon (1:00 PM - 4:00 PM): Picnic & Nature Stroll

**Activity: Affordable Picnic Lunch & Baylands Park Exploration**
*   **Location:** Sunnyvale Baylands Park, 999 E Caribbean Dr, Sunnyvale, CA
*   **Cost:** ~$6 vehicle entry fee (March-October), free for pedestrians/bicycles. (Pack your own lunch to keep food costs down!)
*   **Details:** Drive over to Sunnyvale Baylands Park for a relaxing afternoon. To keep it affordable, pack a delicious picnic lunch from home to enjoy at one of the park's many picnic areas. This expansive park features nature trails winding through protected wetlands, making it a fantastic spot for birdwatching and a peaceful stroll. The flat trails are easy to navigate and offer a refreshing connection with nature.

### 🌅 Evening (4:30 PM - 8:00 PM): Sunset Views & Affordable Dinner

**Activity: Sunset Watching at Palo Alto Baylands Nature Preserve**
*   **Location:** Palo Alto Baylands Nature Preserve, 2575 Embarcadero Rd, Palo Alto, CA (or continue from Sunnyvale Baylands Park, which is nearby)
*   **Cost:** Free
*   **Details:** As evening approaches, head to the Palo Alto Baylands Nature Preserve, known for its incredible sunset views. The flat landscape of the wetlands allows for a wide, unobstructed vista of the sky as the sun dips below the horizon, painting the clouds with vibrant colors. Find a peaceful spot along the trails to relax and enjoy the serene end to your day.

**Activity: Affordable Dinner in Mountain View or Palo Alto**
*   **Location:** Downtown Mountain View or Palo Alto (Castro Street or University Avenue)
*   **Cost:** Affordable (aim for $15-$25 per person)
*   **Details:** For dinner, venture into nearby Mountain View or Palo Alto, which offer a variety of budget-friendly eateries. Consider options like **Curry Up Now** (modern Indian street food), **Mediterranean Wraps** (tasty and filling), or a casual spot for pho or tacos. Many delis and markets, such as **Rose International Market** in Mountain View, also offer hot bars or prepared meals that are affordable and delicious. The "Too Good To Go" app can also offer grab-bag deals from local restaurants and bakeries at a reduced price.

Enjoy your relaxing and artsy day trip!

--------------------------------------------------

